# Session 1: Introduction to LangChain as an Orchestrator

This notebook covers the transition from raw API SDKs (Google GenAI / OpenAI) to using LangChain as a unified orchestrator. We will cover:
1. **Direct SDK comparison** (OpenAI vs. Gemini API client).
2. **LangChain Chat Models** & message structures (`SystemMessage`, `HumanMessage`, `AIMessage`).
3. **State Management**: Building a manual conversational memory loop.
4. **Behind the Scenes**: Enabling LangChain debugging and verbose logging.
5. **Model Configurations**: Setting up parameters like temperature and max tokens.

## Setup & Installation

Install the necessary dependencies and configure your environment variables. This setup is compatible with both local `.env` files and Google Colab Secrets.

In [ ]:
# !pip install openai langchain langchain-openai python-dotenv


In [20]:
import os
from dotenv import load_dotenv

load_dotenv()

if "OPENROUTER_API_KEY" not in os.environ:
    try:
        from google.colab import userdata
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
MODEL_NAME = os.environ.get("MODEL_NAME", "google/gemini-2.0-flash-001")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

print(f"Using OpenRouter model: {MODEL_NAME}")


Using Gemini model: gemini-2.5-flash


## 1. Direct SDKs Comparison

### Google GenAI Client

In [21]:
from google import genai
from google.genai import types

sysmsg = "Imagine you're a travel planner, answer carefully"

# Google GenAI SDK
client = genai.Client()
response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Write one line on AI.",
    config=types.GenerateContentConfig(system_instruction=sysmsg)
)
print(response.text)

AI intelligently crafts your perfect journey, personalizing every detail for an effortless and unforgettable adventure.


### OpenAI Client

In [ ]:
from openai import OpenAI

# OpenAI SDK (using gpt-4o-mini)
client_openai = OpenAI()
completion = client_openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": sysmsg},
        {"role": "user", "content": "Write one line on AI."}
    ],
)
print(completion.choices[0].message.content)

## 2. Using LangChain as an Orchestrator

LangChain abstracts these SDKs into a unified interface, allowing you to swap backends easily without altering the rest of your pipeline.

In [22]:
from langchain_openai import ChatOpenAI

# Using Gemini via LangChain
llm = ChatOpenAI(openai_api_key=OPENROUTER_API_KEY, openai_api_base=OPENROUTER_BASE_URL, model=MODEL_NAME)
response = llm.invoke("Explain LangChain in simple words.")
print(response.content)


Imagine a **Large Language Model (LLM)** like ChatGPT is a super-smart, creative brain. It can write, summarize, answer questions, and generate ideas. But, like a brain in a jar, it has some limitations:

1.  **It doesn't have current information:** Its knowledge stops at its last training cut-off.
2.  **It can't access your specific data:** It doesn't know about your company's documents, your personal emails, or your database.
3.  **It can't use tools:** It can't browse the internet, use a calculator, send an email, or interact with other software.
4.  **It doesn't remember past conversations very well:** Each prompt is often a fresh start.

**LangChain is like giving that super-smart LLM brain a "body," "senses," and the ability to "use tools."**

It's a **developer's toolkit** that makes it much easier to build applications where an LLM can:

*   **Connect to different data sources:** So it can answer questions based on *your* documents, a live database, or the latest news from the 

### Structured Message Inputs

LangChain models receive lists of message objects (`SystemMessage`, `HumanMessage`, `AIMessage`).

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

system_prompt = SystemMessage(content="Imagine you're a travel planner, answer carefully in one line.")
user_message = HumanMessage(content="Tell me about Paris.")

response = llm.invoke([system_prompt, user_message])
print(response.content)

## 3. Stateful Conversations: Manual Loop

By passing the conversational history list along with each new invocation, we can maintain chat history manually.

In [ ]:
from langchain_core.messages import AIMessage

chat_history = []
system_prompt = SystemMessage(content="Imagine you're a travel planner, answer in one line.")

print("Type 'exit' to stop the loop.\n")
while True:
    user_query = input("User: ")
    if user_query.lower() == "exit":
        break
        
    user_message = HumanMessage(content=user_query)
    
    # Pack system prompt, history, and the new message
    full_payload = [system_prompt] + chat_history + [user_message]
    response = llm.invoke(full_payload)
    
    # Append messages to update state
    chat_history.append(user_message)
    chat_history.append(AIMessage(content=response.content))
    
    print(f"AI: {response.content}\n")

## 4. Behind the Scenes: Global Debugging

Debugging in LangChain opens up the "black box" to let you inspect exactly what is happening during a run.

By setting `langchain.debug = True`, LangChain will print a detailed trace of everything it does under the hood for every model call.

In [24]:
import logging
import sys

# Configure python logging to print LangChain debug traces to the screen
logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger("langchain").setLevel(logging.INFO)

In [31]:
from langchain_core.globals import set_debug

# 1. Turn global debug ON
set_debug(True)

# 2. Run your model call (this will now print the full trace below)
print("--- Running with Debug ---\n")
response = llm.invoke("What is 2 + 2?")
print(f"\nAI Response: {response.content}\n")

# 3. Turn global debug OFF
set_debug(False)
print("--- Debug Disabled ---")





--- Running with Debug ---

[llm/start] [llm:ChatGoogleGenerativeAI] Entering LLM run with input:
{
  "prompts": [
    "Human: What is 2 + 2?"
  ]
}
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 429 Too Many Requests"
INFO:google_genai._api_client:Retrying google.genai._api_client.BaseApiClient._request_once in 1.51 seconds as it raised ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 1.249237957s.', 'status': 'RESOURCE_EXHAUSTE

### What does this debug output teach you?

When you run the cell above, look at the logs printed *between* `--- Running with Debug ---` and `AI Response:`:

1. **`[chain/start]` / `[llm/start]`**: Shows when your call starts executing.
2. **`prompts`**: Shows the exact raw string formatted and sent to the Gemini API.
3. **`[llm/end]`**: Shows the token counts (`input_tokens`, `output_tokens`) and metadata returned by Gemini before LangChain extracts the final text response.

## 5. Model Parameters & Configurations

You can fine-tune LLM responses by passing configurations such as `temperature` (randomness control), `max_output_tokens`, and parameters specifically bound to the model during initialization.

In [32]:
# Instantiate the model with customized parameters
strict_model = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0.1,         # Lower temperature results in more factual, reproducible answers
    max_output_tokens=50     # Limit output length
)

response = strict_model.invoke("List three primary colors in bullet points.")
print(response.content)

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 429 Too Many Requests"
INFO:google_genai._api_client:Retrying google.genai._api_client.BaseApiClient._request_once in 1.98 seconds as it raised ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 14.363184339s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 39.136166452s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT